In [14]:
import numpy as np
import cv2
import mediapipe as mp


In [2]:
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

In [3]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR
    return image, results

In [4]:
def draw_landmarks(image, results):
    # Draw face connections
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION)
    # Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
    # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    # Draw right hand connections
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

In [9]:
def draw_styled_landmarks(image, results):
    # Draw face connections
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION,
                              mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
                              mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
                              )
    # Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
                              mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                              )
    # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
                              mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                              )
    # Draw right hand connections
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                              mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                              )

In [22]:
import cv2
cap = cv2.VideoCapture(0)
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
  while cap.isOpened():
    # Read feed
    ret, frame = cap.read()


    image, results = mediapipe_detection(frame, holistic)
    print(results)

    draw_styled_landmarks(image, results)
    # Show to screen
    cv2.imshow('OpenCV Feed', image)
    if cv2.waitKey(10) & 0xFF == ord('q'):
      break
  cap.release()
  cv2.destroyAllWindows()

<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.soluti

In [23]:
# Extract keypoints
len(results.pose_landmarks.landmark)

33

TypeError: len() takes exactly one argument (0 given)

In [ ]:
def extract_keypoints(results):
    # Face: 468 points. Use NaN if not detected
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]) if results.face_landmarks else np.full((468, 3), np.nan)
    
    # Hands: 21 points each
    left_hand = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]) if results.left_hand_landmarks else np.full((21, 3), np.nan)
    right_hand = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]) if results.right_hand_landmarks else np.full((21, 3), np.nan)
    
    # Pose: 33 points (only x,y,z)
    pose = np.array([[res.x, res.y, res.z] for res in results.pose_landmarks.landmark]) if results.pose_landmarks else np.full((33, 3), np.nan)
    
    # Return structured array (543, 3)
    return np.concatenate([face, left_hand, pose, right_hand], axis=0)


In [ ]:
LIPS_IDXS = [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291, 146, 91, 181, 84, 17, 314, 405, 321, 375, 78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308]
LEFT_HAND_IDXS = np.arange(468, 489)
RIGHT_HAND_IDXS = np.arange(522, 543)
LEFT_POSE_IDXS = [500, 502, 504, 506, 508]
RIGHT_POSE_IDXS = [501, 503, 505, 507, 509]

def standardize_and_clean(frames_list):
    """
    Takes a list of (543, 3) arrays, identifies dominant hand, 
    mirrors if needed, and returns a clean (Frames, 92, 3) array.
    """
    data = np.array(frames_list) # Shape: (Frames, 543, 3)
    
    # 1. Identify Dominant Hand
    lh_exists = np.sum(~np.isnan(data[:, LEFT_HAND_IDXS, 0]))
    rh_exists = np.sum(~np.isnan(data[:, RIGHT_HAND_IDXS, 0]))
    left_dominant = lh_exists >= rh_exists
    
    # 2. Extract and Re-order
    lips = data[:, LIPS_IDXS, :]
    if left_dominant:
        hand_1, hand_2 = data[:, LEFT_HAND_IDXS, :], data[:, RIGHT_HAND_IDXS, :]
        pose = data[:, LEFT_POSE_IDXS + RIGHT_POSE_IDXS, :]
    else:
        hand_1 = data[:, RIGHT_HAND_IDXS, :] * [-1, 1, 1]
        hand_2 = data[:, LEFT_HAND_IDXS, :] * [-1, 1, 1]
        lips = lips * [-1, 1, 1]
        pose = data[:, RIGHT_POSE_IDXS + LEFT_POSE_IDXS, :] * [-1, 1, 1]
        
    clean_data = np.concatenate([lips, hand_1, hand_2, pose], axis=1)
    return np.nan_to_num(clean_data).astype(np.float32)


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def visualize_standardized_points(clean_data, frame_idx=0):
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')
    p = clean_data[frame_idx]
    ax.scatter(p[0:40, 0], p[0:40, 1], p[0:40, 2], c='red', label='Lips')
    ax.scatter(p[40:61, 0], p[40:61, 1], p[40:61, 2], c='blue', label='Dom Hand')
    ax.scatter(p[61:82, 0], p[61:82, 1], p[61:82, 2], c='green', label='Support')
    ax.scatter(p[82:92, 0], p[82:92, 1], p[82:92, 2], c='black', label='Pose')
    ax.view_init(elev=-90, azim=-90)
    plt.show()


In [ ]:
# 🧪 TEST THE VISUALIZATION
# Run your OpenCV feed first, then run this cell to see the standardized output!
if 'results' in locals():
    # 1. Extract
    raw_kp = extract_keypoints(results)
    # 2. Clean/Standardize
    clean_kp = standardize_and_clean([raw_kp])
    print(f"Original Landmarks: {raw_kp.shape}")
    print(f"Standardized Subset: {clean_kp.shape}")
    # 3. Plot
    visualize_standardized_points(clean_kp)
else:
    print("Please run the 'OpenCV Feed' cell above first to capture some results!")

In [ ]:
import os
DATA_PATH = os.path.join('MP_Data')
actions = np.array(['hello', 'thanks', 'iloveyou'])
no_sequences = 30
sequence_length = 30

In [ ]:
#hello
##0
##1
##...
##29(no_of_sequences)
#thanks

#iloveyou

In [ ]:

for action in actions:
    for sequence in range(no_sequences):
        try:
            os.makedirs(os.path.join(DATA_PATH, action, str(sequence)))
        except:
            pass

In [ ]:
import cv2
cap = cv2.VideoCapture(0)
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    # Loop through actions
    for action in actions:
        # Loop through sequences aka videos
        for sequence in range(no_sequences):
            # Loop through video length aka sequence length
            for frame_num in range(sequence_length):
                # Read feed
                ret, frame = cap.read()


                image, results = mediapipe_detection(frame, holistic)
                print(results)

                draw_styled_landmarks(image, results)

                keypoints = extract_keypoints(results)
                npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
                np.save(npy_path, keypoints)

                 
                # Show to screen
                cv2.imshow('OpenCV Feed', image)


                if cv2.waitKey(10) & 0xFF == ord('q'):
                    break


    cap.release()
    cv2.destroyAllWindows()

array([ 0.63542527,  0.56462735, -0.04651637, ...,  0.73097378,
        0.41492736,  0.03105556])